In [37]:
import sys
import numpy as np

import geopandas as gpd
import rasterio
from rasterio.mask import mask

import time
import calendar

import pystac_client
from pystac_client.stac_api_io import APIError
from rasterio.errors import RasterioIOError
import planetary_computer
from IPython.display import clear_output


sys.path.append("../utils")

In [38]:
inspections = gpd.read_file(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_master_training_geometries.geojson"
).drop(columns = 'apn')
inspections.head()


,inspection_id,Date,year,month,status,geometry
0,1,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37..."
1,2,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37..."
2,3,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37..."
3,4,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37..."
4,5,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37..."


In [39]:
# # Initial Test Cell

# inspections = inspections.to_crs(epsg=4326)

# catalog = pystac_client.Client.open(
#     "https://planetarycomputer.microsoft.com/api/stac/v1",
#     modifier=planetary_computer.sign_inplace,
# )

# row = inspections.iloc[0]
# geom = row.geometry

# # 3) Compute the bbox
# minx, miny, maxx, maxy = geom.bounds

# # 3) Search June 2021 over that bbox
# search = catalog.search(
#     collections=["sentinel-2-l2a"],
#     bbox=[minx, miny, maxx, maxy],
#     datetime="2021-06-01/2021-06-30",
#     query={"eo:cloud_cover": {"lte": 5}},
# )


# items = search.item_collection()
# print(f"Returned {len(items)} Items")
# items

# item = items[0]

# with rasterio.open(item.assets["B04"].href) as src:
#     raster_crs = src.crs

# # now reproject your inspections into the raster’s CRS
# inspections = inspections.to_crs(raster_crs)
# single_row = single_row.to_crs(raster_crs)

# red_href = item.assets["B04"].href
# nir_href = item.assets["B08"].href

# with rasterio.open(red_href) as src_red, rasterio.open(nir_href) as src_nir:
#     red_clip, red_transform = mask(src_red, single_row.geometry, crop=True)
#     nir_clip, nir_transform = mask(src_nir, single_row.geometry, crop=True)


# # Convert to float32 to avoid integer division
# red = red_clip.astype("float32")
# nir = nir_clip.astype("float32")

# # Compute NDVI = (NIR − Red) / (NIR + Red)
# ndvi = (nir - red) / (nir + red)

# print(f"NDVI array shape:", ndvi.shape)

# mean_ndvi = np.nanmean(ndvi)
# mean_ndvi

# single_row.loc[row.name, "mean_ndvi"] = mean_ndvi
# single_row

# print(f"Mean NDVI for inspection {row.name}: {mean_ndvi:.4f}")

In [40]:
# # First Iteration of Function

# import calendar
# import pystac_client
# import planetary_computer
# import rasterio
# from rasterio.mask import mask
# import numpy as np


# def add_mean_ndvi(dataframe):
#     # work on a copy of the user’s original
#     df_out = dataframe.copy()
#     df_query = df_out.to_crs(epsg=4326)

#     catalog = pystac_client.Client.open(
#         "https://planetarycomputer.microsoft.com/api/stac/v1",
#         modifier=planetary_computer.sign_inplace,
#     )

#     mean_vals = []

#     for idx, row in df_query.iterrows():
#         geom = row.geometry

#         # build month window
#         year, month = int(row["year"]), int(row["month"])
#         last_day = calendar.monthrange(year, month)[1]
#         start = f"{year}-{month:02d}-01"
#         end = f"{year}-{month:02d}-{last_day:02d}"

#         # search
#         minx, miny, maxx, maxy = geom.bounds
#         search = catalog.search(
#             collections=["sentinel-2-l2a"],
#             bbox=[minx, miny, maxx, maxy],
#             datetime=f"{start}/{end}",
#             query={"eo:cloud_cover": {"lte": 5}},
#         )
#         items = search.item_collection()

#         if not items:
#             # no scenes found → record NaN and move on
#             mean_vals.append(np.nan)
#             continue

#         item = items[0]

#         # get COG CRS
#         with rasterio.open(item.assets["B04"].href) as src:
#             cog_crs = src.crs

#         # reproject this one geometry into the COG CRS
#         single = df_query.loc[[idx]].to_crs(cog_crs)
#         mask_geom = [single.geometry.iloc[0]]

#         # clip & compute NDVI
#         with (
#             rasterio.open(item.assets["B04"].href) as src_red,
#             rasterio.open(item.assets["B08"].href) as src_nir,
#         ):
#             red_clip, _ = mask(src_red, mask_geom, crop=True)
#             nir_clip, _ = mask(src_nir, mask_geom, crop=True)

#         red = red_clip.astype("float32")
#         nir = nir_clip.astype("float32")
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ndvi = (nir - red) / (nir + red)

#         mean_vals.append(float(np.nanmean(ndvi)))

#     # attach back to the original dataframe
#     df_out["mean_ndvi"] = mean_vals
#     return df_out


In [41]:
# # Second Iteration of Function

# import calendar
# import pystac_client
# import planetary_computer
# import rasterio
# from rasterio.mask import mask
# import numpy as np


# def add_mean_ndvi(dataframe):

#     # Copy the user's original dataframe to avoid messing up their original
#     df_out = dataframe.copy()
#     df_query = df_out.to_crs(epsg=4326)

#     # Open the catalog
#     catalog = pystac_client.Client.open(
#         "https://planetarycomputer.microsoft.com/api/stac/v1",
#         modifier=planetary_computer.sign_inplace,
#     )

#     # Declare an empty list for the mean ndvi values
#     mean_vals = []

#     # Iterate over the dataframe by inspection_id
#     for insp_id in df_query["inspection_id"]:
#         # pull the row by its inspection_id
#         row = df_query[df_query["inspection_id"] == insp_id].iloc[0]
#         geom = row.geometry

#         # Build date range based on dataframe 'year' and 'month' columns
#         year, month = int(row["year"]), int(row["month"])
#         last_day = calendar.monthrange(year, month)[1]
#         start = f"{year}-{month:02d}-01"
#         end = f"{year}-{month:02d}-{last_day:02d}"

#         # Search for overlapping scenes, by inspection geometry
#         try:
#             search = catalog.search(
#                 collections=["sentinel-2-l2a"],
#                 bbox=geom.bounds,
#                 datetime=f"{start}/{end}",
#                 query={"eo:cloud_cover": {"lte": 5}},
#             )
#             item = next(search.items(), None)

#         except APIError as e:
#             print(f"inspection_id {insp_id}: STAC APIError, skipping → NaN")
#             mean_vals.append(np.nan)
#             # optional backoff
#             time.sleep(0.5)
#             continue

#         # If no scenes found, make this NaN and continue
#         if not item:
#             mean_vals.append(np.nan)
#             print(f"inspection_id {insp_id}: no scene found, NDVI set to NaN")
#             continue
        

#         # Get the scene's CRS to match the geometry to
#         with rasterio.open(item.assets["B04"].href) as src:
#             cog_crs = src.crs

#         # Reproject the geometry to that CRS
#         single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
#         mask_geom = [single.geometry.iloc[0]]

#         try:
#             with (
#                 rasterio.open(item.assets["B04"].href) as src_red,
#                 rasterio.open(item.assets["B08"].href) as src_nir,
#             ):
#                 red_clip, _ = mask(src_red, mask_geom, crop=True)
#                 nir_clip, _ = mask(src_nir, mask_geom, crop=True)

#         except ValueError as e:
#             # this happens if shapes really don't overlap
#             mean_vals.append(np.nan)
#             print(f"inspection_id {insp_id}: geometry did not overlap, NDVI set to NaN")
#             continue

#         red = red_clip.astype("float32")
#         nir = nir_clip.astype("float32")
        
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ndvi = (nir - red) / (nir + red)

#         mean_val = float(np.nanmean(ndvi))
#         mean_vals.append(mean_val)
#         print(f"inspection_id {insp_id}: NDVI calculated = {mean_val:.4f}")

#     # attach back to the original dataframe in the same order
#     df_out["mean_ndvi"] = mean_vals
#     return df_out


In [42]:
# # Fourth Iteration, Cloud Cover Adjusting cell

# def add_mean_ndvi(dataframe, cloud_thresh=0.20):
#     """
#     For each inspection polygon, find the first Sentinel-2 L2A scene in the given month,
#     check that < cloud_thresh fraction of pixels are cloud (using the SCL band),
#     then compute mean NDVI or record NaN.
#     """
#     df_out = dataframe.copy()
#     df_query = df_out.to_crs(epsg=4326)

#     catalog = pystac_client.Client.open(
#         "https://planetarycomputer.microsoft.com/api/stac/v1",
#         modifier=planetary_computer.sign_inplace,
#     )

#     mean_vals = []

#     for insp_id in df_query["inspection_id"]:
#         row = df_query[df_query["inspection_id"] == insp_id].iloc[0]
#         geom = row.geometry

#         # build date window
#         year, month = int(row["year"]), int(row["month"])
#         last_day = calendar.monthrange(year, month)[1]
#         start = f"{year}-{month:02d}-01"
#         end = f"{year}-{month:02d}-{last_day:02d}"

#         # STAC search (scene-level cloud cover ≤ cloud_thresh*100)
#         try:
#             search = catalog.search(
#                 collections=["sentinel-2-l2a"],
#                 bbox=geom.bounds,
#                 datetime=f"{start}/{end}",
#                 query={"eo:cloud_cover": {"lte": int(cloud_thresh * 100)}},
#             )
#             item = next(search.items(), None)
#         except APIError:
#             print(f"inspection_id {insp_id}: STAC APIError → NaN")
#             mean_vals.append(np.nan)
#             time.sleep(0.5)
#             continue

#         if item is None:
#             print(f"inspection_id {insp_id}: no scene found → NaN")
#             mean_vals.append(np.nan)
#             continue

#         # get the COG CRS
#         with rasterio.open(item.assets["B04"].href) as src:
#             cog_crs = src.crs

#         # reproject geometry
#         single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
#         mask_geom = [single.geometry.iloc[0]]

#         # — NEW: load the SCL band and compute cloud fraction within the AOI —
#         try:
#             with rasterio.open(item.assets["SCL"].href) as src_scl:
#                 scl_clip, _ = mask(src_scl, mask_geom, crop=True)
#         except Exception:
#             print(f"inspection_id {insp_id}: SCL missing or error → NaN")
#             mean_vals.append(np.nan)
#             continue

#         scl = scl_clip[0].astype("uint8")
#         # SCL codes 7 through 11 represent cloud classes
#         cloud_mask = np.isin(scl, [7, 8, 9, 10, 11])
#         frac_cloud = cloud_mask.sum() / cloud_mask.size

#         if frac_cloud > cloud_thresh:
#             print(f"inspection_id {insp_id}: {frac_cloud:.2%} cloudy → NaN")
#             mean_vals.append(np.nan)
#             continue

#         # clip red & NIR, compute NDVI
#         try:
#             with (
#                 rasterio.open(item.assets["B04"].href) as src_red,
#                 rasterio.open(item.assets["B08"].href) as src_nir,
#             ):
#                 red_clip, _ = mask(src_red, mask_geom, crop=True)
#                 nir_clip, _ = mask(src_nir, mask_geom, crop=True)
#         except ValueError:
#             print(f"inspection_id {insp_id}: geometry no overlap → NaN")
#             mean_vals.append(np.nan)
#             continue

#         red = red_clip.astype("float32")
#         nir = nir_clip.astype("float32")
#         with np.errstate(divide="ignore", invalid="ignore"):
#             ndvi = (nir - red) / (nir + red)

#         mean_val = float(np.nanmean(ndvi))
#         mean_vals.append(mean_val)
#         print(
#             f"inspection_id {insp_id}: NDVI = {mean_val:.4f} ({frac_cloud:.2%} clouds)"
#         )

#     df_out["mean_ndvi"] = mean_vals
#     return df_out


In [43]:
# Third Iteration of Function


def add_mean_ndvi(dataframe):
    df_out = dataframe.copy()

    df_query = df_out.to_crs(epsg=4326)

    # Open the catalog once
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

    mean_vals = []

    for insp_id in df_query["inspection_id"]:
        row = df_query[df_query["inspection_id"] == insp_id].iloc[0]
        geom = row.geometry

        # Build date range
        year, month = int(row["year"]), int(row["month"])
        last_day = calendar.monthrange(year, month)[1]
        start = f"{year}-{month:02d}-01"
        end = f"{year}-{month:02d}-{last_day:02d}"

        # Search but at NaN if API Error is found
        # Search built from month and year columns in dataframe
        # Searching for imagery with <25% cloud cover. <5% wasn't catching many scenes.
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=geom.bounds,
                datetime=f"{start}/{end}",
                query={"eo:cloud_cover": {"lte": 25}},
            )
            item = next(search.items(), None)

        except APIError:
            print(f"inspection_id {insp_id}: STAC API Timeout Error: skipping, NaN")
            mean_vals.append(np.nan)
            time.sleep(0.5)
            continue

        if item is None:
            print(f"inspection_id {insp_id}: no scene found, NaN")
            mean_vals.append(np.nan)
            continue

        # Get CRS of raster
        with rasterio.open(item.assets["B04"].href) as src:
            cog_crs = src.crs

        # Reproject geometry
        single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
        mask_geom = [single.geometry.iloc[0]]

        # Clip and compute NDVI, avoiding errors. If error, input NaN
        try:
            with (
                rasterio.open(item.assets["B04"].href) as src_red,
                rasterio.open(item.assets["B08"].href) as src_nir,
            ):
                red_clip, _ = mask(src_red, mask_geom, crop=True)
                nir_clip, _ = mask(src_nir, mask_geom, crop=True)
        except (ValueError, RasterioIOError) as e:
            print(f"inspection_id {insp_id}: masking error ({e}), NaN")
            mean_vals.append(np.nan)
            continue

        # Calculate NDVI, suppressing warnings.
        red = red_clip.astype("float32")
        nir = nir_clip.astype("float32")
        with np.errstate(divide="ignore", invalid="ignore"):
            ndvi = (nir - red) / (nir + red)

        mean_val = float(np.nanmean(ndvi))
        mean_vals.append(mean_val)
        print(f"inspection_id {insp_id}: NDVI calculated = {mean_val:.4f}")

    df_out["mean_ndvi"] = mean_vals
    return df_out


In [44]:
# Dataframe split; running the whole frame takes way too long. We'll do it in tenths.

# total rows
n = len(inspections)

# compute a chunk size so that the first 9 are equal and the last picks up any remainder
chunk_size = n // 10
remainder = n % 10

splits = []
start = 0
for i in range(10):
    extra = 1 if i < remainder else 0
    stop = start + chunk_size + extra
    splits.append(inspections.iloc[start:stop])
    start = stop

# now print out the sizes
for i, split_df in enumerate(splits, start=1):
    print(f"Split {i}: {len(split_df)} rows, {len(split_df.columns)} columns")

Split 1: 6758 rows, 6 columns
Split 2: 6758 rows, 6 columns
Split 3: 6758 rows, 6 columns
Split 4: 6758 rows, 6 columns
Split 5: 6758 rows, 6 columns
Split 6: 6758 rows, 6 columns
Split 7: 6758 rows, 6 columns
Split 8: 6758 rows, 6 columns
Split 9: 6758 rows, 6 columns
Split 10: 6758 rows, 6 columns


In [45]:
# Maybe you already processed some splits! Select the split to start from here.
start_split = 4

for i, split_df in enumerate(splits[start_split - 1 :], start=start_split):
    clear_output(wait=True)
    print(f"--- Processing split {i} of {len(splits)} ---")

    # Compute NDVI for this chunk
    df_chunk = add_mean_ndvi(split_df)

    # Report how many NaNs were produced
    n_missing = df_chunk["mean_ndvi"].isna().sum()
    
    # Count how many mean_ndvi values are below -0.1 (likely clouds)
    n_clouds = (df_chunk["mean_ndvi"] < -0.1).sum()
    print(
        f"Split {i}: {n_missing} missing NDVI values; "
        f"{n_clouds} values below -0.1 (likely clouds)"
    )
    
    # Save only the needed columns
    df_chunk = df_chunk[["inspection_id", "mean_ndvi"]]

    # Save to CSV, embedding the split number in the filename
    out_path = (
        f"/capstone/wildfire_prep/ryan/data-preparation/code/07_ndvi_pystac/ndvi_files/ndvi_{i:02d}_of10.csv"
    )
    df_chunk.to_csv(out_path, index=False)


--- Processing split 5 of 10 ---
inspection_id 27033: NDVI calculated = 0.3723
inspection_id 27034: NDVI calculated = 0.2738
inspection_id 27035: NDVI calculated = 0.4425
inspection_id 27036: NDVI calculated = 0.3491
inspection_id 27037: NDVI calculated = 0.1736
inspection_id 27038: NDVI calculated = 0.3902
inspection_id 27039: NDVI calculated = 0.3383
inspection_id 27040: NDVI calculated = 0.2466
inspection_id 27041: NDVI calculated = 0.2501
inspection_id 27042: NDVI calculated = 0.3317
inspection_id 27043: NDVI calculated = 0.3873
inspection_id 27044: NDVI calculated = 0.2099
inspection_id 27045: NDVI calculated = 0.3079
inspection_id 27046: NDVI calculated = 0.3932
inspection_id 27047: NDVI calculated = 0.3155
inspection_id 27048: NDVI calculated = 0.4274
inspection_id 27049: NDVI calculated = 0.3189
inspection_id 27050: NDVI calculated = 0.2995
inspection_id 27051: NDVI calculated = 0.2167
inspection_id 27052: NDVI calculated = 0.3685
inspection_id 27053: NDVI calculated = 0.3361
i

KeyboardInterrupt: 